# How-To: Build a Sparse Workflow

The sparse backend computes the same properties as the dense pipeline — band structure, DOS and PDOS, Boltzmann transport — for systems whose Hamiltonian is too large to hold in memory as dense arrays. What you give up in exchange is exactness: the backend keeps only the hoppings that matter and solves only the eigenstates a property needs. Both are *controlled* approximations with a clear physical meaning, and both are yours to set.

This How-To goes through them one at a time.

**What you will learn**

- What the sparse backend approximates, and what it keeps exact
- What `threshold` and `rcut` each truncate, and what each costs in band energies
- How `energy_window` and `interior_window` restrict the solve to the energy range you care about, and what physics that leaves out
- What `hk_solver` chooses, and when the choice matters to you
- How to check a sparse result before trusting it
- Where cell doubling fits in — the case this backend was built for

**Prerequisites**: [Tutorial 01](../tutorial01.ipynb) — the dense workflow this one mirrors call for call.


## 1. What the sparse backend does

The dense pipeline holds the real-space Hamiltonian $H_{\alpha\beta}(\mathbf{R})$ as one array over (orbital, orbital, lattice vector), and then builds $H(\mathbf{k})$ and $\partial H/\partial\mathbf{k}$ on the *entire* k-mesh at once. Memory therefore scales as $n_\mathrm{awf}^2 n_k$, and $n_\mathrm{awf}$ — the number of atomic orbitals — is set by how many atoms are in your cell.

The sparse backend changes three things:

1. **$H(\mathbf{R})$ is stored as a list of hoppings**, not as an array. Hoppings whose amplitude is negligible are never stored at all.
2. **$H(\mathbf{k})$ is built one k-point at a time** and thrown away before the next. No quantity indexed by both $\mathbf{k}$ and two orbital indices ever exists.
3. **Only the eigenstates a property needs are solved for**, rather than the whole spectrum at every k-point.


| | |
| --- | --- |
| **Exact** | the Fourier interpolation $H(\mathbf{R}) \rightarrow H(\mathbf{k})$ at any $\mathbf{k}$, including interpolation onto a finer mesh; the velocities $\partial\varepsilon_n/\partial\mathbf{k}$; the property formulas themselves — adaptive-smearing DOS and the Boltzmann transport integrals are the *same code* the dense pipeline runs |
| **Approximate** | which hoppings are kept (`threshold`, `rcut`); how many eigenstates are solved (`energy_window`, `interior_window`) |

So every approximation in a sparse run comes from one of four flags, and each one has a physical statement behind it. The rest of this How-To is those four flags, plus the practicalities around them.

**Reach for the sparse backend when** the dense pipeline runs out of memory on the system you need. **Stay dense when** it fits: the dense pipeline is exact, has every PAOFLOW feature, and asks you to make no truncation decisions at all.


## 2. Switching a script to the sparse backend

`SparsePAOFLOW` mirrors the dense driver's method names and signatures, so a working dense script becomes a sparse one in two lines — the import, and the constructor. A dense script:

```python
from PAOFLOW import PAOFLOW

paoflow = PAOFLOW.PAOFLOW(savedir='silicon.save', outputdir='output', smearing='gauss')
paoflow.read_atomic_proj_QE()
paoflow.projectability()
paoflow.pao_hamiltonian()
paoflow.bands(ibrav=2, nk=2000)
paoflow.interpolated_hamiltonian(nfft1=24, nfft2=24, nfft3=24)
paoflow.pao_eigh()
paoflow.gradient_and_momenta()
paoflow.adaptive_smearing()
paoflow.dos(emin=-12.0, emax=2.2, ne=1000)
paoflow.transport(emin=-12.0, emax=2.2)
paoflow.finish_execution()
```

and the same calculation on the sparse backend:

```python
from PAOFLOW.SparsePAOFLOW import SparsePAOFLOW                     # <-- changed

paoflow = SparsePAOFLOW(savedir='silicon.save', outputdir='output_sparse',
                        smearing='gauss', threshold=1.0e-4)          # <-- changed
paoflow.read_atomic_proj_QE()
paoflow.projectability()
paoflow.pao_hamiltonian()
paoflow.bands(ibrav=2, nk=2000)
paoflow.interpolated_hamiltonian(nfft1=24, nfft2=24, nfft3=24)
paoflow.pao_eigh()
paoflow.gradient_and_momenta()
paoflow.adaptive_smearing()
paoflow.dos(emin=-12.0, emax=2.2, ne=1000)
paoflow.transport(emin=-12.0, emax=2.2)
paoflow.finish_execution()
```

Reading the input files, checking projectability and building the PAO Hamiltonian are the same dense operations in both drivers. They act on your cell before any supercell construction, where the Hamiltonian is small, and the switch to sparse storage happens at the end of `pao_hamiltonian()`.

One difference in behaviour to expect. `pao_eigh()`, `gradient_and_momenta()` and `adaptive_smearing()` do not compute anything on their own here: eigenvalues, velocities and adaptive smearing widths all come out of a **single pass over the k-mesh**, triggered by the first property that needs them (`dos()` or `transport()`). The calls stay in the script so dense and sparse scripts remain line-for-line comparable, but the time will be reported against the mesh pass rather than against them.

::: {note}
A method with no sparse counterpart raises `NotImplementedError`. Use the dense driver for that property.
:::


## 3. What is stored: the hoppings

A PAO Hamiltonian in real space *is* a tight-binding model: orbital $\alpha$ in the home cell couples to orbital $\beta$ in the cell displaced by lattice vector $\mathbf{R}$, with amplitude $H_{\alpha\beta}(\mathbf{R})$ in eV. The sparse backend stores exactly that — for each surviving hopping, the two orbitals, the lattice vector, the amplitude, and the intra-cell separation of the two orbitals (which is what $\partial H/\partial\mathbf{k}$, and hence the band velocity, needs).

After the conversion the run reports what it kept. On the silicon example (18 orbitals, $12^3$ k-grid, `threshold=1e-4`):

```
Sparse H(R): nawf=18, nR=1728, nnz=0.353M, density=6.30e-01, mem=17.5 MB
             (dense equivalent 8.5 MB), truncation eig-shift bound=5.56e-01 eV
```

- **`nnz`** — surviving hoppings.
- **`density`** — the fraction of the dense array retained. 63 % here, which is *not* a mistake and not unusual. $H(\mathbf{R})$ built this way is the exact discrete Fourier transform of $H(\mathbf{k})$ on the DFT k-mesh, not a maximally localised Wannier Hamiltonian, so it has real weight at large $|\mathbf{R}|$. Section 5 explains why that matters.
- **`mem` against `dense equivalent`** — at this size the hopping list is *larger* than the array it replaces, because a hopping carries its indices along with its value. The sparse backend is not a win for small cells and is not meant to be; what it changes is how the cost grows (Section 10).
- **`eig_bound`** — a rigorous bound on how far truncation can move *any* eigenvalue at *any* k-point. How to read it is the subject of the next section.


## 4. `threshold` — drop weak hoppings

```python
SparsePAOFLOW(..., threshold=1.0e-4)   # eV
```

`threshold` is the amplitude in eV below which a hopping $|H_{\alpha\beta}(\mathbf{R})|$ is discarded. It is the primary control, and the physical statement is simply that couplings weaker than this do not measurably affect the bands you care about.

Measured on the silicon example:

| `threshold` (eV) | hoppings kept | fraction | `eig_bound` (eV) |
| ---------------- | ------------- | -------- | ---------------- |
| 0 (none)  | 559 872 | 100 %  | 0     |
| $10^{-6}$ | 542 980 | 97.0 % | 0.0004 |
| $10^{-5}$ | 523 544 | 93.5 % | 0.0066 |
| $10^{-4}$ | 352 836 | 63.0 % | 0.556 |
| $10^{-3}$ |  77 652 | 13.9 % | 5.52  |
| $10^{-2}$ |  19 190 |  3.4 % | 16.5  |

### Reading `eig_bound`

The discarded part of the Hamiltonian is itself Hermitian at every $\mathbf{k}$, and Gershgorin's theorem bounds its spectral norm by its largest absolute row sum. A perturbation of that norm can move an eigenvalue by at most that much, so `eig_bound` is a **rigorous worst case**, valid at every k-point, computed without diagonalising anything.

It is also a very loose one. Diagonalising the truncated and untruncated Hamiltonians at the same k-points, the largest eigenvalue shift at `threshold=1e-4` is **6 meV**, against an `eig_bound` of **556 meV** — about a hundredfold pessimistic, which is typical.

**So use it as a safety bound, not an accuracy estimate.** A small `eig_bound` means you are certainly fine. A large one means you have to measure.

### What it costs in practice

End to end against a dense reference on the silicon example:

| `threshold` | band error | verdict |
| ----------- | ---------- | ------- |
| $10^{-3}$ | ~160 meV | too coarse for anything quantitative |
| $10^{-4}$ | ~10 meV  | **a good default** |
| $10^{-5}$ | ~9 meV   | no measurable gain over $10^{-4}$, more memory |

The residual ~10 meV at $10^{-4}$ is not truncation error and tightening `threshold` will not remove it. Away from the original DFT k-grid, the sparse and dense codes interpolate through slightly different Hermitisation conventions; on the grid points themselves they agree exactly. It is a floor, and it sits well below the accuracy of the underlying DFT.

**Practical rule.** Start at $10^{-4}$. Tighten it only if a sweep on *your* material shows the property you care about still moving; loosen it only after measuring what that costs. How much a given threshold discards depends on how localised your material's $H(\mathbf{R})$ is, so the numbers above do not transfer between systems.


## 5. `rcut` — drop long-range hoppings

```python
SparsePAOFLOW(..., threshold=1.0e-4, rcut=45.0)   # Bohr; default None
```

`rcut` is a **second and physically different** truncation: it discards hoppings by physical bond length $|a_\mathrm{lat}\mathbf{R} + \boldsymbol{\tau}_\alpha - \boldsymbol{\tau}_\beta|$ in Bohr, rather than by amplitude. The statement behind it is that the interaction is short-ranged — a claim about the material, not about numerical noise.

The two truncations interact, so a run with `rcut` set is **not comparable** with one without it, and a threshold sweep no longer characterises the truncation on its own. `eig_bound` covers both. The default `rcut` is `None`.

On the silicon example, at `threshold=1e-4`, the compression looks attractive:

| `rcut` (Bohr) | hoppings kept | vs none | `eig_bound` (eV) |
| ------------- | ------------- | ------- | ---------------- |
| none | 352 836 | 1.00$\times$ | 0.56 |
| 60   | 329 454 | 1.07$\times$ | 0.86 |
| 50   | 306 954 | 1.15$\times$ | 1.11 |
| 45   | 281 032 | 1.26$\times$ | 1.44 |
| 40   | 233 980 | 1.51$\times$ | 2.22 |
| 30   | 110 870 | 3.18$\times$ | 4.92 |
| 20   |  37 066 | 9.52$\times$ | 12.9 |

The accuracy does not. Against a dense reference over a $[-12, 2.2]$ eV property window:

| `rcut` (Bohr) | max band error | DOS $L_1$ error |
| ------------- | -------------- | --------------- |
| none | **19.5 meV** | 1.24 % |
| 50   | 24.2 meV | 1.26 % |
| 45   | 28.8 meV | 1.38 % |
| 40   | 46.9 meV | 1.31 % |
| 30   | 323 meV  | 3.20 % |
| 20   | 645 meV  | 12.2 % |



- **DOS is far more forgiving than bands.** At `rcut=30` the DOS is off by 3 % while individual bands are off by 323 meV — Brillouin-zone integration averages the shifts away. Validating `rcut` on a DOS plot alone would hide the band error completely. **Validate on bands.**
- `eig_bound` tracks the trend but is again ~100$\times$ pessimistic, so it cannot substitute for the comparison.

**Practical rule.** Leave `rcut=None`. Reach for it only after the energy window (Section 6) has failed to make the calculation fit, and only after repeating this comparison on your own material.

::: {note}
`rcut` is applied together with `threshold` when the Hamiltonian is first converted, and cannot be applied later — the bond geometry it needs is no longer recoverable once a supercell has been built (Section 10).
:::


## 6. Restricting the solve to an energy range

The third approximation is not about the Hamiltonian at all — it is about how much of the spectrum you ask for.

By default the backend solves for as many bands per k-point as the dense pipeline would carry, which is set by how many bands were projectable onto the PAO basis. For a property that lives in a finite energy window — DOS or transport within a few $k_BT$ of $E_F$ — most of those states never enter the answer. Two flags exploit that, in different ways and with very different consequences.

### 6.1 `energy_window` — solve from the bottom up to your window

```python
paoflow.energy_window(emin=-12.0, emax=2.2, margin=1.0)
```

This solves for the lowest states only, enough of them to cover everything up to `emax + margin`. The count is estimated by diagonalising at a small set of k-points across the zone and padding the largest count found; coverage is then checked at every k-point during the run, so a window that turns out too short stops the run and tells you the count to use, rather than silently truncating your spectrum.

| argument | meaning |
| -------- | ------- |
| `emin`, `emax` | the property range you will actually plot |
| `margin=1.0` | eV of headroom above `emax` |
| `nprobe=16` | k-points used to estimate the count |
| `nev=None` | give an integer to set the count directly and skip the estimate |

**The margin is physics, not padding.** Two things reach above `emax`: the adaptive smearing width, which gives every state a finite spread (widths here are $\lesssim 0.22$ eV, so four standard deviations is $\lesssim 0.9$ eV), and the occupation derivative $-\partial f/\partial\varepsilon$ in the transport integrals, which is a few $k_BT$ wide (~0.1 eV at 300 K). A margin of 1.0 eV covers both at room temperature. Raise it for higher temperatures or a coarser mesh.

**One caveat.** The number of bands written to `bands_*.dat` becomes "bands inside the window" rather than "all projectable bands", so a windowed run's band file has a different number of columns from an unwindowed one and the two are not column-comparable. DOS and transport are unaffected — the normalisations do not depend on the count.

**What it cannot do.** It cannot make the eigenproblem cheap for a metal or a narrow-gap semiconductor. A window measured from the bottom of the spectrum must still reach $E_F$, which for any real material means 20–50 % of all states. That fraction does not improve as the cell grows.

### 6.2 `interior_window` — solve inside a window

```python
paoflow.interior_window(elo=-3.0, ehi=3.0)
```

This solves only the states *inside* $[\varepsilon_\mathrm{lo}, \varepsilon_\mathrm{hi}]$, never touching the states below. That is the mode where an iterative eigensolver genuinely pays: a narrow window around $E_F$ holds a small fraction of the spectrum no matter how large the cell is.

It pays most for a Hamiltonian that is genuinely sparse in $\mathbf{k}$-space — a large cell whose hopping range is short compared with its size, such as a moiré supercell. Measured on such a system, for 20 states inside the window:

| orbitals | $H(\mathbf{k})$ density | interior solve | full diagonalisation | speedup |
| -------- | ----------------------- | -------------- | -------------------- | ------- |
| 1 600  | 0.81 %  | 0.07 s  | 0.91 s   | 13.7$\times$ |
| 3 600  | 0.36 %  | 0.22 s  | 11.3 s   | 50.2$\times$ |
| 8 100  | 0.16 %  | 0.71 s  | 136.6 s  | 192$\times$  |
| 57 600 | 0.023 % | 12.13 s | —        | —            |

It helps much less for a small cell repeated many times, where the hopping range is comparable to the supercell and $H(\mathbf{k})$ stays nearly dense. It is still worth using there, because asking for 3 states instead of 256 is a large saving regardless of which kernel wins.

**What an interior window permanently removes.** The states below $\varepsilon_\mathrm{lo}$ are never computed, so the electron count is gone with them. Anything fixed by charge neutrality cannot be evaluated. Each consequence produces a warning and the run continues to the next property, with every skipped property restated at the end of the run:

| property | behaviour |
| -------- | --------- |
| DOS / PDOS | range **clamped** to the window; skipped entirely if it does not overlap |
| transport | chemical-potential scan clamped to at least $10k_BT$ inside each edge |
| Hall coefficient | **skipped** — it needs the total carrier count |
| band files | padded with `NaN`; columns are no longer band indices, since the number of states inside the window varies from k-point to k-point |

**Edge contamination is the subtle one.** Adaptive smearing gives every state a finite width, so a state just *below* $\varepsilon_\mathrm{lo}$ — one this mode never computes — would still have contributed intensity inside the window. Measured on a coarse mesh with 2 eV of clearance, this was a **10 % error in the local DOS**. `smear_margin_eV` (default 0.5 eV) clamps the plotted range away from both edges, and the widths actually encountered are checked against it during the run.

::: {warning}
Adaptive smearing widths scale as $n_k^{-1/3}$. On a $4^3$ mesh they reach ~3.9 eV — wide enough that **no useful interior window exists at all**. This mode needs a converged k-mesh before it means anything.
:::

| argument | meaning |
| -------- | ------- |
| `elo`, `ehi` | the window actually solved |
| `kT_margin_eV=0.26` | transport margin per side; $10k_BT$ at 300 K — raise it for higher $T$ |
| `smear_margin_eV=0.5` | DOS/PDOS margin per side, against edge contamination |

The two window modes are **mutually exclusive**: one solves from the bottom of the spectrum, the other never looks below its lower edge.


## 7. `hk_solver` — which eigensolver, per k-point

```python
SparsePAOFLOW(..., hk_solver='auto')   # 'auto' | 'sparse' | 'dense'
```

::: {important}
This names the **eigensolver used at a single k-point, and nothing else**. `'dense'` does not put the run back on the dense pipeline: $H(\mathbf{R})$ remains a hopping list, $H(\mathbf{k})$ is still built one k-point at a time, and nothing indexed by the whole k-mesh is ever formed. The only dense object is one matrix for the current k-point, released before the next.
:::

- **`'sparse'`** — an iterative (Lanczos) solve that never forms $H(\mathbf{k})$ as a matrix.
- **`'dense'`** — a direct diagonalisation of the current k-point.
- **`'auto'`** (default) — chooses between them from the size of the problem, and reports the choice.

**Leave it on `'auto'`.** An iterative solve only pays when you want a small fraction of the spectrum; asking it for half the eigenvalues costs more memory and more time than diagonalising outright. `'auto'` uses the iterative kernel when the requested fraction is below roughly an eighth, and diagonalises directly otherwise. Above ~4000 orbitals *and* a large requested fraction it stops instead of silently consuming the node, and tells you to narrow the energy window.


## 8. PDOS


### `plan_pdos` — ask for PDOS before the mesh runs

Eigenvalues, velocities, smearing widths and PDOS weights are all produced in one streaming pass over the k-mesh, and the eigenvectors for each k-point are discarded as soon as that k-point is done. A PDOS accumulation can therefore only join **while the pass is running**.

Asking for PDOS after some other property has already triggered the mesh forces a complete second pass — roughly doubling the run time. The fix is free: call `plan_pdos(emin, emax, ne)` before the first property, or simply call `dos()` before `transport()`.

::: {warning}
PDOS writes one file per orbital, so a large supercell produces one file per orbital in that supercell. Pass `do_pdos=False` to `dos()` unless you want them.
:::


## 9. Checking a sparse result

Every flag above trades accuracy for memory, so a sparse number is only as good as the check behind it. In rough order of cost:

1. **Read the run's log.** The sparse-specific diagnostics go to `<outputdir>/sparse.log`: the flags in force, the hopping-list statistics and `eig_bound`, the energy-window sizing, and which eigensolver was chosen. Everything that changes a run's cost — a second mesh pass for PDOS, a clamped or skipped property under an interior window — is also printed as you run.

2. **Sweep `threshold` and look at bands, not DOS.** Run at $10^{-3}$, $10^{-4}$, $10^{-5}$ and compare band energies inside your property window. As Section 5 showed, a DOS plot can look converged while individual bands are off by hundreds of meV.

3. **Compare against a dense reference** on the smallest system that still shows the physics you care about. This is the check that matters most, and it is worth constructing a small case for. 

4. **Cross-check the two eigensolvers** on that same small system: `hk_solver='sparse'` against `hk_solver='dense'`, everything else identical.

5. **If you enabled `rcut`, re-derive it** for your own material against a dense reference. The exchange rate in Section 5 is a property of silicon's $H(\mathbf{R})$, not a constant.



## 10. An example where sparse is required: Cell doubling


```python
paoflow.doubling_Hamiltonian(nx=1, ny=1, nz=1)   # 2x2x2 = 8 primitive cells
```

`doubling_Hamiltonian` builds a supercell by doubling along each lattice vector, folding the Brillouin zone as it goes. It is how PAOFLOW reaches large cells — for defects, disorder, superlattices, or any question that needs a cell larger than the DFT calculation used.

**`nx`, `ny`, `nz` are doubling counts, not cell multipliers.** The multiplier is $N = 2^{n_x+n_y+n_z}$, so `(2, 2, 2)` is a 64-cell supercell — eight times larger than `(1, 1, 1)`.

### Why the dense pipeline fails here, and the sparse one does not

The number of orbitals grows by $N$, and the $\mathbf{R}$ grid stays fixed, so **dense storage grows as $N^2$** — four times per doubling. The hopping list grows as $N$: doubling replicates each hopping exactly twice, because a hopping in the small cell becomes the same hopping in each of the two copies. Two, not four.

For the silicon example:

| doublings | $N$ | orbitals | dense $H(\mathbf{R})$ | dense $\partial H/\partial\mathbf{k}$ | hopping list |
| --------- | --- | -------- | --------------------- | ------------------------------------- | ------------ |
| none    | 1  | 18   | 8.5 MB  | 25.6 MB | 17.5 MB |
| 1, 1, 1 | 8  | 144  | 547 MB  | 1.6 GB  | 167 MB  |
| 2, 2, 2 | 64 | 1152 | 34.2 GB | 103 GB  | 1.6 GB  |

At `(2, 2, 2)` the dense pipeline needs 34 GB for the Hamiltonian alone and another 103 GB for its gradient; the sparse one needs 1.6 GB. 

Before allocating anything, the sparse driver projects the cost of the doubling you asked for and refuses sizes that cannot fit, so a job fails quickly with a number rather than as an out-of-memory kill an hour into a job. `mem_budget_gb` states the budget explicitly if the machine reports its memory badly; `force=True` overrides the refusal.

::: {note}
The hopping list is held in full by every MPI rank. Running **fewer ranks per node** is therefore a way to make a large calculation fit, which is the opposite of the usual intuition.
:::


### A supercell script

```python
from PAOFLOW.SparsePAOFLOW import SparsePAOFLOW

paoflow = SparsePAOFLOW(
    savedir='silicon.save',
    outputdir='output_sparse_222',
    smearing='gauss',
    verbose=True,
    threshold=1.0e-4,   # calibrated against a dense reference on the primitive cell
    rcut=None,          # second truncation axis; not calibrated here, so off
    hk_solver='auto',
)
paoflow.read_atomic_proj_QE()
paoflow.projectability()
paoflow.pao_hamiltonian()

paoflow.doubling_Hamiltonian(nx=2, ny=2, nz=2)           # N = 64, 1152 orbitals
paoflow.energy_window(emin=-12.0, emax=2.2, margin=1.0)  
paoflow.bands(ibrav=2, nk=500)                              
paoflow.interpolated_hamiltonian(nfft1=6, nfft2=6, nfft3=6)  # = 24^3 primitive sampling
paoflow.pao_eigh()
paoflow.gradient_and_momenta()
paoflow.adaptive_smearing()
paoflow.dos(emin=-12.0, emax=2.2, ne=1000, do_pdos=False)    # 1152 orbital files otherwise
paoflow.transport(emin=-12.0, emax=2.2)

paoflow.finish_execution()
```


## 11. Troubleshooting


| message | meaning | what to do |
| ------- | ------- | ---------- |
| `SparsePAOFLOW does not implement 'X'` | that property has no sparse counterpart | use the dense driver for it |
| `Projected peak exceeds the budget` | the requested cell cannot fit; nothing was allocated | reduce the doubling count, set `rcut`, run fewer ranks per node, or set `mem_budget_gb` |
| `past the iterative regime, but n exceeds dense_n_max` | too many orbitals *and* too large a fraction of the spectrum requested | narrow the range with `energy_window` |
| `energy_window() ran before doubling_Hamiltonian()` | ordering; the count would be scaled by the cell multiplier | move the call after the doubling |
| `interior_window() is already active` | the two window modes are mutually exclusive | pick one |
| `requested emax exceeds the lowest computed top band` | the solved bands do not span the transport range | widen the window |
| `Sparse mesh is being re-run from scratch to accumulate PDOS` | PDOS was requested after the mesh already ran; costs a second full pass | call `plan_pdos()` first, or `dos()` before `transport()` |
| `may be contaminated near the interior-window edges` | smearing tails reach past the window edge, where nothing was computed | widen the window or raise `smear_margin_eV` |
| `SKIPPED under interior_window` | the property needs states the window never computed | use `energy_window` for that property |


